# SQL scaffolding for the deliberately dirty 3NF source database

This notebook scaffolds the PostgreSQL/Neon portion of the data-engineering exercise. The database is intentionally **dirty** so the downstream Pandas workflow can practice profiling, cleaning, merging/entity resolution, transformations, mini-aggregations, serialization, feature engineering, and scaling.

The current exercise assumes that **`companies` already exists and contains data**. The setup below preserves that table and recreates/populates the other teaching tables.


# Learning objectives

By completing this notebook, you will be able to:

- read a relational schema and identify primary keys and relationships;
- explain why data is separated into reference/lookup tables;
- execute PostgreSQL DDL and DML from Python;
- use `BEGIN`, `COMMIT`, and `ROLLBACK`;
- distinguish database integrity from data quality;
- deliberately construct a dirty source dataset;
- identify missing values, inconsistent labels, duplicate-looking records, and suspicious numeric values;
- use SQL to profile and locate data-quality problems before using Pandas;
- preserve the raw database extract as the starting point for downstream cleaning.

The extended workshop describes the downstream progression as **quick profiling → spotting the grime → finding the grime → transformations → mini-aggregations → serialization checkpoint**.


# 1. The use case

NorthStar AI Technologies Inc. has collected employee information from several operational systems. The information is useful, but different systems used different conventions:

```text
Artificial Intelligence
AI
artificial intelligence

Toronto
toronto
TORONTO

Remote
remote
REMOTE
WFH

Machine Learning Engineer
ML Engineer
Machine Learning Eng.
```

The source also contains missing values, placeholders, duplicate-looking employees, conflicting records, inconsistent identifiers, and suspicious numerical values.

### Your job

Do not make the database clean. Create a realistic **dirty source** that students can later investigate and clean.

> The dirt is intentional. The reasoning required to clean it is the learning objective.


# 2. Relational model

The teaching database contains:

```text
companies
departments
locations
work_arrangements
job_levels
positions
data_sources
wage_benchmarks
employees
```

Conceptually:

```text
companies
    |
    | 1:M
    v
employees
    |
    +---- M:1 ---- positions ---- M:1 ---- job_levels
                         |
                         +---- M:1 ---- departments
    |
    +---- M:1 ---- locations
    |
    +---- M:1 ---- work_arrangements

data_sources
    |
    | 1:M
    v
wage_benchmarks
```

The intended unit of observation in `employees` is **one employee source record**. This does not mean every row is a unique real-world employee; duplicate and near-duplicate records are part of the exercise.


# 3. Transactional SQL

The construction script is wrapped in:

```sql
BEGIN;
...
COMMIT;
```

If execution fails, Python performs `ROLLBACK`. This creates a controlled boundary around database construction.

`companies` is intentionally **not** dropped or recreated because it already contains data. The other teaching tables are disposable and are recreated by the script.


In [ ]:
import os
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host=os.getenv("NEON_HOST"),
    port=os.getenv("NEON_PORT", "5432"),
    dbname=os.getenv("NEON_DATABASE"),
    user=os.getenv("NEON_USER"),
    password=os.getenv("NEON_PASSWORD"),
    sslmode="require"
)

print("Connected to PostgreSQL.")


# 4. Verify `companies` before changing anything

Run this checkpoint first. The setup script assumes that the `companies` table already contains the NorthStar company record(s).


In [ ]:
companies_before = pd.read_sql_query(
    """
    SELECT company_id, company_name, industry, country
    FROM companies
    ORDER BY company_id;
    """,
    conn
)
companies_before


# 5. Full dirty-database SQL

Read the SQL before executing it. Identify the intentional grime and explain why the `employees` source deliberately has no foreign-key constraints.

The following is also the **single copyable SQL block** that can be pasted into a PostgreSQL/Neon SQL editor.


In [ ]:
DIRTY_DATABASE_SQL = "BEGIN;\nDROP TABLE IF EXISTS employees CASCADE;\nDROP TABLE IF EXISTS wage_benchmarks CASCADE;\nDROP TABLE IF EXISTS positions CASCADE;\nDROP TABLE IF EXISTS job_levels CASCADE;\nDROP TABLE IF EXISTS work_arrangements CASCADE;\nDROP TABLE IF EXISTS locations CASCADE;\nDROP TABLE IF EXISTS departments CASCADE;\nDROP TABLE IF EXISTS data_sources CASCADE;\n\nCREATE TABLE departments (department_id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY, department_name VARCHAR(100));\nINSERT INTO departments (department_name) VALUES ('Artificial Intelligence'),('AI'),('artificial intelligence'),('Data Science'),('DATA SCIENCE'),('Data science '),('Software Engineering'),('Software Eng.'),('software engineering'),('Robotics'),('ROBOTICS'),('Research & Development'),('R&D'),('Research and Development'),('Human Resources'),('HR'),('Human Resources '),('Finance'),('FINANCE'),('Sales & Marketing'),('Sales and Marketing'),('Sales'),('Marketing'),('Innovation'),('Corporate Strategy'),('AI Innovation'),('Software Development');\n\nCREATE TABLE locations (location_id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY, city VARCHAR(100), province VARCHAR(100), country VARCHAR(100));\nINSERT INTO locations (city, province, country) VALUES ('Toronto','Ontario','Canada'),('toronto','ON','Canada'),('TORONTO','Ontario','CANADA'),('Kitchener','Ontario','Canada'),('Kitchener','ON','Canada'),('Kitchener, ON',NULL,'Canada'),('Waterloo','Ontario','Canada'),('Waterloo','ON','Canada'),('Waterloo','Ontario',NULL),('Ottawa','Ontario','Canada'),('OTTAWA','ON','Canada'),('Montreal','Quebec','Canada'),('Montréal','Québec','Canada'),('Montreal','QC','Canada'),('Vancouver','British Columbia','Canada'),('Vancouver','BC','Canada'),('Calgary','Alberta','Canada'),('Calgary','AB','Canada'),('Edmonton','Alberta','Canada'),('Edmonton','AB','Canada'),('Mississauga','Ontario','Canada'),('Hamilton','Ontario','Canada'),('Markham','ON','Canada');\n\nCREATE TABLE work_arrangements (work_arrangement_id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY, arrangement_name VARCHAR(100));\nINSERT INTO work_arrangements (arrangement_name) VALUES ('Remote'),('remote'),('REMOTE'),('Work From Home'),('WFH'),('Hybrid'),('hybrid'),('HYBRID'),('Hybrid '),('On-site'),('On site'),('Onsite'),('on-site'),('Office');\n\nCREATE TABLE job_levels (job_level_id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY, level_name VARCHAR(100), level_rank INTEGER);\nINSERT INTO job_levels (level_name, level_rank) VALUES ('Junior',1),('Jr',1),('junior',1),('Intermediate',2),('Mid-Level',2),('mid level',2),('Senior',3),('Sr',3),('senior',3),('Lead',4),('Team Lead',4),('Manager',5),('Mgr',5),('Director',6),('Dir',6);\n\nCREATE TABLE positions (position_id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY, position_title VARCHAR(150), department_id INTEGER, job_level_id INTEGER, noc_code VARCHAR(20));\nINSERT INTO positions (position_title, department_id, job_level_id, noc_code) VALUES ('Machine Learning Engineer',1,4,'21211'),('ML Engineer',2,5,'21211'),('machine learning engineer',3,6,'21211'),('Machine Learning Eng.',1,7,NULL),('Data Scientist',4,4,'21211'),('DATA SCIENTIST',5,5,'21211'),('Data scientist ',6,6,NULL),('Software Developer',7,1,'21232'),('Software Developer',7,2,'21232'),('software developer',8,3,'21232'),('Software Dev.',9,4,NULL),('Robotics Engineer',10,4,'21399'),('Robotics Engineer',11,5,'21399'),('Robotics Eng.',12,6,NULL),('Research Engineer',13,4,'21211'),('Research Engineer',14,5,'21211'),('HR Specialist',15,4,'11200'),('Human Resources Specialist',16,5,'11200'),('Financial Analyst',18,4,'11101'),('Finance Analyst',19,5,NULL),('Sales Specialist',20,4,'62100'),('Sales Rep',22,5,'64101'),('Marketing Specialist',23,4,'11202'),('Marketing Specialist',20,5,NULL);\n\nCREATE TABLE data_sources (source_id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY, source_name VARCHAR(200), publisher VARCHAR(150), reference_period VARCHAR(50), source_url TEXT, notes TEXT);\nINSERT INTO data_sources (source_name,publisher,reference_period,source_url,notes) VALUES ('Job Bank Wage Report','Government of Canada','2025','https://www.jobbank.gc.ca/','Canadian labour market wage benchmark'),('Job Bank','Gov. of Canada','2025','https://www.jobbank.gc.ca/',NULL),('Labour Force Survey','Statistics Canada','2025','https://www.statcan.gc.ca/','Labour market statistics'),('Labour Force Survey ','Statistics Canada','2025','https://www.statcan.gc.ca/',NULL);\n\nCREATE TABLE wage_benchmarks (benchmark_id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY, source_id INTEGER, noc_code VARCHAR(20), occupation_name VARCHAR(150), city VARCHAR(100), low_hourly NUMERIC(8,2), median_hourly NUMERIC(8,2), high_hourly NUMERIC(8,2));\nINSERT INTO wage_benchmarks (source_id,noc_code,occupation_name,city,low_hourly,median_hourly,high_hourly) VALUES (1,'21211','Machine Learning Engineer','Toronto',30,48,75),(2,'21211','ML Engineer','toronto',31,49,76),(1,'21211','Machine Learning Engineer','Waterloo',29,46,72),(2,'21211','Machine Learning Engineer','Waterloo',NULL,47,73),(1,'21211','Data Scientist','Toronto',27,44,68),(2,'21211','DATA SCIENTIST','Toronto',28,45,69),(1,'21232','Software Developer','Toronto',25,42,65),(2,'21232','Software Dev.','toronto',26,43,66),(1,'21399','Robotics Engineer','Toronto',28,45,70),(1,'21399','Robotics Eng.','Toronto',NULL,45,70);\n\nCREATE TABLE employees (employee_id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY, company_id INTEGER, first_name VARCHAR(100), last_name VARCHAR(100), department_id INTEGER, position_id INTEGER, location_id INTEGER, work_arrangement_id INTEGER, start_date DATE, experience_months INTEGER, salary INTEGER);\nINSERT INTO employees (company_id,first_name,last_name,department_id,position_id,location_id,work_arrangement_id,start_date,experience_months,salary) VALUES\n(1,'Alice','Chen',1,1,1,6,'2022-03-14',36,105000),(2,' alice ','CHEN',2,2,2,7,'2022-03-14',36,106000),(1,' bob ','SMITH',3,3,3,1,'2021-06-01',60,142000),(1,'Bob','Smith',NULL,2,4,NULL,NULL,NULL,140000),(1,'CARLOS','Martinez ',1,1,1,6,'2019-09-23',96,158000),(1,'Carlos','Martínez',3,1,2,2,'2019-09-23',100,160000),(1,'Diana','Patel',1,4,4,10,'2018-01-08',120,178000),(1,'Emily','Wong',1,1,1,2,'2023-07-17',18,82000),(1,'Emily','Wong ',1,1,1,2,'2023-07-17',18,NULL),(1,'farah','KHAN',1,2,2,1,'2020-11-02',72,137500),(1,'George','Brown',1,3,3,2,'2017-04-10',144,165000),(1,'Hannah ',' O''Brien',1,1,1,2,'2024-02-12',6,78000),(1,'Ian','Wilson',4,5,1,2,'2022-05-09',30,97000),(1,'Julia','Garcia',4,6,1,1,'2020-03-16',72,131000),(1,'Kevin','LEE',5,6,2,2,'2019-08-19',84,129500),(1,'Kevin','Lee',4,7,8,7,'2019-08-19',84,130000),(1,'Laura','Nguyen',4,7,3,2,'2021-01-11',60,112000),(1,'Michael','Johnson',4,5,1,1,'2023-09-04',12,91000),(1,'Nadia','Ali',4,6,5,2,'2020-10-26',72,134000),(1,'Oliver','Martin',4,7,1,3,'2022-11-14',24,104000),(1,'Priya','Shah',4,5,2,2,'2024-01-15',3,60000),(1,'Quentin','Brown',7,8,1,2,'2023-02-06',12,72000),(1,'Rachel','Taylor',7,9,2,2,'2021-05-17',48,98000),(1,'sam','MILLER',7,10,1,1,'2018-07-30',108,145000),(1,'Thomas','Anderson',7,11,4,3,'2016-02-22',180,185000),(1,'Uma','Thomas',7,8,1,2,'2024-06-03',0,68000),(1,'Victor ','Garcia',7,9,3,1,'2022-08-08',36,101000),(1,'Wendy','Clark',7,10,1,2,'2019-12-02',84,132000),(1,'Xavier','Davis',10,12,1,10,'2020-01-13',84,108000),(1,'Yasmin','Rahman',10,13,2,7,'2018-09-17',120,149000),(1,'Zachary','Wilson',11,14,1,1,'2017-06-05',156,171000),(1,'anna','MARTIN',10,12,3,2,'2023-10-23',18,85000),(1,'Ben','Lee',11,13,6,1,'2021-03-29',60,115000),(1,'Chloe','  Davis',10,14,1,2,'2022-12-05',30,118000),(1,'Daniel','Moore',13,15,1,2,'2021-07-12',48,103000),(1,'Eva','MORGAN',14,16,4,1,'2019-04-15',96,141000),(1,'Frank','King',13,15,2,2,'2023-03-20',12,88000),(1,'Grace','Lee',14,16,1,3,'2016-11-07',180,155000),(1,'Helen','Scott',15,17,1,2,'2020-08-31',72,85000),(1,'Isaac','Young',16,18,3,3,'2023-04-24',12,72000),(1,'Jessica','Adams',18,19,1,2,'2021-09-13',48,92000),(1,'Kyle','Baker',18,20,2,1,'2024-04-08',6,64000),(1,'Linda','Nelson',20,21,1,2,'2019-02-11',96,88000),(1,'Mark','Carter',22,21,1,10,'2022-06-20',36,76000),(1,'Nora','Mitchell',23,22,5,1,'2020-11-23',72,81000),(1,'Oscar','Perez',20,23,6,2,'2023-08-14',12,69000),(1,NULL,'Roberts',7,9,1,2,'2022-04-18',24,93000),(1,'Unknown','Unknown',7,9,1,2,'2022-05-16',24,95000),(1,'','N/A',4,5,1,2,'2023-01-09',12,88000),(1,'Jordan','King',NULL,5,1,2,'2022-09-12',36,99000),(1,'Morgan','Hill',4,NULL,1,2,'2021-10-18',48,105000),(1,'Taylor','Scott',7,9,NULL,2,'2020-06-15',72,110000),(1,'Casey','Young',10,12,1,NULL,'2022-02-14',36,98000),(1,'Jamie','Adams',2,5,1,2,'2022-07-11',36,NULL),(1,'Alex','Turner',1,2,1,2,'2023-01-16',300,135000),(1,'Cameron','White',3,9,1,2,'2023-05-15',12,199999),(1,'Jordan','Brown',1,4,1,1,'2018-06-18',144,61000);\n\nDO $$\nBEGIN\n    IF NOT EXISTS (SELECT 1 FROM companies) THEN RAISE EXCEPTION 'Expected existing companies data was not found.'; END IF;\n    IF NOT EXISTS (SELECT 1 FROM employees) THEN RAISE EXCEPTION 'Employees were not populated.'; END IF;\nEND $$;\n\nCOMMIT;\n"

print("SQL characters:", len(DIRTY_DATABASE_SQL))


# 6. Execute the transaction

The SQL contains `BEGIN` and `COMMIT`. The exception handler performs `ROLLBACK` if anything fails.


In [ ]:
try:
    with conn.cursor() as cur:
        cur.execute(DIRTY_DATABASE_SQL)
    conn.commit()
    print("Dirty database created successfully.")
except Exception as exc:
    conn.rollback()
    print("Transaction rolled back.")
    print("Error:", exc)
    raise


# 7. Verify the table inventory

A successful execution is not enough. Confirm that the expected tables exist.


In [ ]:
expected_tables = [
    "companies", "departments", "locations", "work_arrangements",
    "job_levels", "positions", "data_sources", "wage_benchmarks", "employees"
]

inventory = pd.read_sql_query(
    """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
      AND table_name = ANY(%s)
    ORDER BY table_name;
    """,
    conn, params=(expected_tables,)
)
inventory


# 8. Verify row counts

Now ask: **Did the SQL create the amount of data we expected?** This is construction validation, not data cleaning.


In [ ]:
counts = []
for table in expected_tables:
    n = pd.read_sql_query(f"SELECT COUNT(*) AS row_count FROM {table};", conn).iloc[0, 0]
    counts.append((table, int(n)))
pd.DataFrame(counts, columns=["table_name", "row_count"])


# 9. Confirm that `companies` was preserved

Compare the current company table with the snapshot taken before construction. The dirty-data setup is supposed to leave `companies` alone.


In [ ]:
companies_after = pd.read_sql_query(
    """
    SELECT company_id, company_name, industry, country
    FROM companies
    ORDER BY company_id;
    """, conn
)
print("Rows before:", len(companies_before))
print("Rows after: ", len(companies_after))
companies_after


# 10. Inspect the dirty lookup tables

Look for capitalization differences, abbreviations, whitespace, semantically equivalent labels, missing values, and duplicate concepts. Do not clean anything yet.


In [ ]:
for table in ["departments", "locations", "work_arrangements", "job_levels", "positions", "data_sources", "wage_benchmarks"]:
    print(f"\n--- {table} ---")
    display(pd.read_sql_query(f"SELECT * FROM {table} ORDER BY 1;", conn))


# 11. Retrieve the raw employee source

This is the handoff from SQL into Pandas. Preserve this DataFrame as `df_raw`; do not overwrite it during cleaning.


In [ ]:
df_raw = pd.read_sql_query(
    """SELECT * FROM employees ORDER BY employee_id;""",
    conn
)
print("Rows:", len(df_raw))
print("Columns:", len(df_raw.columns))
df_raw.head(10)


# 12. SQL quick profiling

Before using Pandas, practice a few SQL profiling queries. The objective is to **spot the grime**, not fix it.


In [ ]:
sql_profile = """
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT employee_id) AS distinct_employee_ids,
    COUNT(*) FILTER (WHERE first_name IS NULL) AS missing_first_name,
    COUNT(*) FILTER (WHERE last_name IS NULL) AS missing_last_name,
    COUNT(*) FILTER (WHERE salary IS NULL) AS missing_salary,
    COUNT(*) FILTER (WHERE department_id IS NULL) AS missing_department,
    COUNT(*) FILTER (WHERE position_id IS NULL) AS missing_position,
    COUNT(*) FILTER (WHERE location_id IS NULL) AS missing_location
FROM employees;
"""
pd.read_sql_query(sql_profile, conn)


In [ ]:
duplicate_candidates = pd.read_sql_query(
    """
    SELECT LOWER(TRIM(first_name)) AS first_name_normalized,
           LOWER(TRIM(last_name)) AS last_name_normalized,
           COUNT(*) AS record_count
    FROM employees
    GROUP BY LOWER(TRIM(first_name)), LOWER(TRIM(last_name))
    HAVING COUNT(*) > 1
    ORDER BY record_count DESC;
    """, conn
)
duplicate_candidates


In [ ]:
suspicious_values = pd.read_sql_query(
    """
    SELECT employee_id, first_name, last_name, experience_months, salary
    FROM employees
    WHERE salary IS NULL
       OR salary < 60000
       OR salary > 200000
       OR experience_months < 0
       OR experience_months > 240
    ORDER BY employee_id;
    """, conn
)
suspicious_values


# 13. Student reasoning checkpoint

Answer these in Markdown.

**Transaction:** Why is a transaction useful here?

**Lookup tables:** Which tables are reference/lookup tables?

**Five forms of grime:** Identify at least five.

**Entity resolution:** Identify one candidate duplicate pair. What evidence supports the match? What evidence conflicts?

**Referential integrity:** Why might a production database use foreign keys while this dirty source does not?

**Provenance:** Which data is synthetic and which is reference/context data?


## Student response — TODO

**Transaction:**


**Lookup tables:**


**Five forms of grime:**
1.
2.
3.
4.
5.

**Candidate duplicate:**


**Evidence:**


**Referential integrity:**


**Provenance:**



# 14. Handoff to the Data Cleaning notebook

The SQL phase is complete. The next workflow is:

```text
PostgreSQL → df_raw → QUICK PROFILE → SPOT THE GRIME → FIND THE GRIME
→ APPLY TRANSFORMATIONS → MINI-AGGREGATIONS → SERIALIZATION CHECKPOINT
→ FEATURE ENGINEERING → SCALING → EDA
```

The key rule is: **Never clean a problem that you have not first demonstrated exists.**


# 15. SQL construction checklist

- [ ] `companies` existed before the script ran.
- [ ] `companies` was not dropped.
- [ ] The SQL was executed transactionally.
- [ ] The expected tables were created.
- [ ] Row counts were verified.
- [ ] `companies` was preserved.
- [ ] Dirty lookup tables were inspected.
- [ ] `employees` was retrieved into `df_raw`.
- [ ] SQL profiling was performed.
- [ ] Candidate duplicates were identified.
- [ ] Suspicious numeric records were identified.
- [ ] No cleaning was performed prematurely.
- [ ] I can explain why the source is intentionally dirty.
